# Linux Basic File Commands and Links (Educational Notebook)
This notebook covers the everyday commands for navigating the filesystem, creating/copying/moving/removing files and directories, viewing file contents, and creating hard and symbolic links.

## 1. Navigating the Filesystem

```bash
pwd              # print working directory (the full path of where you currently are)
cd /etc            # change directory to an absolute path
cd ..                # move up one directory
cd -                   # jump back to the previous directory
cd                       # with no argument, go to your home directory
cd ~                       # same thing, explicitly

ls                  # list the contents of the current directory
ls /var               # list the contents of a specific directory
ls -l                   # "long" format: permissions, owner, group, size, modification date
ls -a                     # include hidden files (names starting with a dot)
ls -la                      # combine both
ls -lh                        # long format with human-readable sizes (K/M/G instead of raw bytes)
```

A path is **absolute** if it starts with `/` (measured from the filesystem root) and **relative** if it doesn't (measured from the current directory). `.` means "this directory" and `..` means "the parent directory".


## 2. Creating and Removing Files and Directories

```bash
touch notes.txt              # create an empty file, or update an existing file's modification time
mkdir project                  # create a directory
mkdir -p project/src/utils       # create a nested path in one step, creating any missing parent directories

rmdir empty_dir                    # remove a directory, but only if it is empty
rm notes.txt                         # remove a file
rm -r project                          # remove a directory and everything inside it, recursively
rm -rf project                           # same, but never prompt and ignore nonexistent files -- use with care
```

`rm` has no undo and no trash can by default - a deleted file is gone. `rm -rf` in particular deletes silently and recursively; always double-check the path (and that you're in the directory you think you're in) before running it, especially with a wildcard.


## 3. Copying, Moving, and Renaming

```bash
cp source.txt dest.txt         # copy a file
cp source.txt dir/               # copy a file into a directory, keeping its name
cp -r src_dir dest_dir              # copy a directory recursively
cp -p source.txt dest.txt             # preserve permissions, ownership, and timestamps while copying

mv old_name.txt new_name.txt       # rename a file (mv is also how you rename, there's no separate "rename" command)
mv file.txt dir/                     # move a file into a directory
mv dir1 dir2                           # rename (or move) a directory
```

`mv` on the same filesystem is typically just a fast metadata update (no data actually copied); moving *across* filesystems requires an actual copy-then-delete, which `mv` handles automatically but which takes noticeably longer for large files.


## 4. Viewing File Contents

```bash
cat file.txt              # print an entire file to the screen (fine for short files)
less file.txt                # view a file one screen at a time, scrollable, searchable with /pattern
more file.txt                  # an older, more limited pager (mostly superseded by less)

head file.txt                    # print the first 10 lines
head -n 20 file.txt                 # print the first 20 lines
tail file.txt                         # print the last 10 lines
tail -n 20 file.txt                     # print the last 20 lines
tail -f /var/log/syslog                   # "follow" a file, printing new lines as they're appended -- useful for watching logs live
```

`cat` is best for short files or for feeding a file into a pipeline; `less` is the right tool once a file is longer than a screenful.


## 5. What Is a Link?

Recall (or preview, if this is new) that every file is represented on disk by an **inode**, which stores its metadata and a pointer to its data blocks - the filename itself is just an entry in a directory that points at an inode number. A **link** is simply another directory entry pointing at that same underlying content.

Linux supports two different kinds of links, with very different behavior: **hard links** and **symbolic (soft) links**.


## 6. Hard Links

A **hard link** is a second directory entry pointing at the *same inode* as an existing file - not a copy, and not a reference to the filename, but a second name for the exact same data.

```bash
ln original.txt hardlink.txt      # create a hard link
ls -li original.txt hardlink.txt     # -i shows the inode number: both names share the same one
```

Key properties:
- Both names are equally "real" - there is no concept of an "original" versus a "copy" at the filesystem level once the link exists.
- Editing the content through either name changes the same underlying data, immediately visible through the other name too.
- The data is only actually freed once **every** hard link to it has been removed (the inode's link count drops to zero) - deleting one name leaves the content intact under the other.
- Hard links cannot cross filesystem/partition boundaries (an inode number is only meaningful within its own filesystem).
- Ordinary users cannot hard-link directories (this is restricted by the kernel to avoid creating loops in the directory tree).


## 7. Symbolic (Soft) Links

A **symbolic link** (or **symlink**) is a small special file that stores a *path*, not an inode number - conceptually similar to a shortcut.

```bash
ln -s /var/log/nginx/access.log latest_log      # create a symbolic link
ls -l latest_log                                    # symlinks are shown with -> pointing at their target
readlink latest_log                                   # print just the target path a symlink points to
```

Key properties:
- A symlink has its own separate inode, distinct from the target's inode.
- It can point to a file *or* a directory, and can cross filesystem/partition boundaries freely.
- If the target is deleted or renamed, the symlink still exists but becomes a **dangling link** - it points at a path that no longer resolves to anything, and attempting to open it fails.
- Ordinary users *can* create symbolic links to directories, unlike hard links.


## 8. Hard Links vs. Symbolic Links

| | Hard link | Symbolic link |
|---|---|---|
| What it stores | The same inode number as the target | A path string pointing at the target |
| Own inode? | No - shares the target's inode | Yes - has its own inode |
| Can link a directory? | No (regular users) | Yes |
| Can cross filesystems? | No | Yes |
| Survives target deletion? | Yes - data lives on as long as any link remains | No - becomes a dangling/broken link |
| How to identify | `ls -li`, matching inode numbers | `ls -l` shows `name -> target` |

As a rule of thumb: use a **symlink** when you want a flexible, human-readable shortcut (and don't mind it breaking if the target moves); a hard link is a much narrower tool, mostly useful for things like keeping multiple names for a file within the same filesystem while guaranteeing the content can't accidentally be left orphaned by deleting just one name.


## Hands-on

Try these on your own system:

```bash
mkdir -p ~/linktest && cd ~/linktest
echo "hello" > original.txt

ln original.txt hardlink.txt
ln -s original.txt symlink.txt

ls -li original.txt hardlink.txt symlink.txt
cat symlink.txt

rm original.txt
cat hardlink.txt          # still works -- the data survived
cat symlink.txt              # fails -- now a dangling link
ls -l symlink.txt              # shows the broken link in a different color/style in most terminals
```

Note what changes (and what doesn't) at each step, and match it back to the properties described above.


## Review Questions

1. What is the difference between an absolute path and a relative path?
2. What does `mkdir -p` do differently from plain `mkdir` when creating a nested path?
3. Why is `rm -rf` considered dangerous, and what precaution should you take before running it?
4. When is `less` a better choice than `cat` for viewing a file?
5. What does it mean for two hard links to "share an inode", and how could you confirm that on the command line?
6. Why can't you create a hard link across two different filesystems, but you can create a symbolic link across them?
7. If you delete the file a hard link points to, what happens to the hard link? What happens to a symbolic link in the same situation?
8. Why are ordinary users prevented from creating hard links to directories?
9. How would you tell, just by looking at `ls -l` output, whether an entry is a symbolic link?


# Cheat Sheet

```
Navigation:
  pwd   cd <path>   cd ..   cd -   cd (home)
  ls   ls -l   ls -a   ls -la   ls -lh

Create/remove:
  touch file        mkdir dir        mkdir -p a/b/c
  rmdir dir (empty only)      rm file      rm -r dir      rm -rf dir

Copy/move:
  cp src dst      cp -r srcdir dstdir      cp -p (preserve attrs)
  mv old new (rename)      mv file dir/ (move)

View contents:
  cat file      less file      head file      tail file      tail -f file

Links:
  ln target hardlink        (same inode, same filesystem, no directories)
  ln -s target symlink        (path-based, cross-filesystem OK, can break)
  ls -li          show inode numbers
  readlink name     show a symlink's target
```
